# 两层网络识别手写数字

Axiom 配套实验。请先读懂对应教材，再按三个阶段练习。计算在你自己的电脑运行。

## 准备环境

建议 Python 3.11 或 3.12，在单独虚拟环境中运行下面的安装命令，然后用 `jupyter notebook` 打开本文件。

```bash
python -m pip install notebook numpy matplotlib scikit-learn
```

先运行“跟着做”。带 TODO 的函数初始返回 None，检查单元会提示待完成。补全后重新运行函数定义与检查单元。独立练习应另写实现，再核对结果。参考解答在文件末尾；网页的阶段勾选是个人学习记录，不代表系统已执行你的代码。

依据斋藤康毅教材第 1—5 章。为降低下载与运行负担，使用 scikit-learn 内置的 8×8 手写数字，而非 MNIST。数据来源：UCI Optical Recognition of Handwritten Digits，经 scikit-learn 提供。只用 NumPy 实现网络，scikit-learn 仅负责提供数据与划分。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits=load_digits()
Xt,Xv,yt,yv=train_test_split(digits.data/16.,digits.target,test_size=.25,random_state=0,stratify=digits.target)
def init_net(inputs,hidden,classes,seed=0):
    rng=np.random.default_rng(seed)
    return {'W1':rng.normal(0,np.sqrt(2/inputs),(inputs,hidden)),'b1':np.zeros(hidden),'W2':rng.normal(0,np.sqrt(2/hidden),(hidden,classes)),'b2':np.zeros(classes)}
def forward(p,X):
    z=X@p['W1']+p['b1'];a=np.maximum(z,0)
    logits=a@p['W2']+p['b2'];ex=np.exp(logits-logits.max(axis=1,keepdims=True))
    return z,a,ex/ex.sum(axis=1,keepdims=True)
def loss(p,X,y):
    prob=forward(p,X)[2]
    return -np.log(prob[np.arange(len(y)),y]+1e-12).mean()
def check_gradients(solver):
    rng=np.random.default_rng(4);X=rng.normal(size=(3,4));y=np.array([0,1,2]);p=init_net(4,5,3,3)
    analytic=solver(p,X,y)
    for key in p:
        numeric=np.zeros_like(p[key])
        for index in np.ndindex(p[key].shape):
            original=p[key][index];p[key][index]=original+1e-5;up=loss(p,X,y)
            p[key][index]=original-1e-5;down=loss(p,X,y);p[key][index]=original
            numeric[index]=(up-down)/2e-5
        assert analytic[key].shape==p[key].shape
        error=np.linalg.norm(analytic[key]-numeric)/(np.linalg.norm(analytic[key])+np.linalg.norm(numeric)+1e-7)
        assert error<1e-4,(key,error)
    print('四组梯度通过小网络数值检查。')
def train(gradient,hidden=32,seed=0,steps=240):
    p=init_net(64,hidden,10,seed);rng=np.random.default_rng(seed);history=[]
    for _ in range(steps):
        index=rng.choice(len(Xt),64,replace=False)
        gradients=gradient(p,Xt[index],yt[index])
        for key in p:p[key]-=.12*gradients[key]
        history.append(loss(p,Xt[index],yt[index]))
    print('测试准确率：',np.mean(forward(p,Xv)[2].argmax(axis=1)==yv))
    return p,history
def check_training(result):
    p,history=result
    assert len(history)>20 and np.isfinite(history).all()
    assert np.mean(history[-20:])<np.mean(history[:20])
    assert forward(p,Xv)[2].shape==(len(Xv),10)
    print('训练流程检查通过；还需分析错误样本。')

## 1. 跟着做

逐步运行示例，解释每个输入、运算和输出。再修改一个输入观察变化。

In [ ]:
def demo_gradients(p,X,y):
    z,a,prob=forward(p,X)
    dlogits=prob.copy();dlogits[np.arange(len(y)),y]-=1;dlogits/=len(y)
    dW2=a.T@dlogits;db2=dlogits.sum(axis=0)
    dz=(dlogits@p['W2'].T)*(z>0)
    return {'W1':X.T@dz,'b1':dz.sum(axis=0),'W2':dW2,'b2':db2}
network,history=train(demo_gradients)
plt.plot(history);plt.xlabel('step');plt.ylabel('loss');plt.show()
predictions=forward(network,Xv)[2].argmax(axis=1)
for index in np.flatnonzero(predictions!=yv)[:4]:
    plt.imshow(Xv[index].reshape(8,8),cmap='gray');plt.title('true=%d predicted=%d'%(yv[index],predictions[index]));plt.show()

## 2. 自己补全

在下面函数中填写 TODO。可回看示例，但请说明你填写的每一行做了什么。

In [ ]:
def student_gradients(p,X,y):
    z,a,prob=forward(p,X)
    dlogits=prob.copy()
    dlogits[np.arange(len(y)),y]-=1
    dlogits/=len(y)
    # TODO 1: 由 a 与 dlogits 求 W2、b2 的梯度
    # TODO 2: 梯度通过 W2 和 ReLU，得到隐藏层梯度
    # TODO 3: 求 W1、b1 的梯度，返回四个同名键的字典
    return None

In [ ]:
if student_gradients(init_net(4,5,3),np.ones((3,4)),np.array([0,1,2])) is None:print('待完成：补全四组梯度。')
else:check_gradients(student_gradients)

## 3. 独立完成

独立用隐藏层宽度 16、随机种子 7 训练，返回参数和损失历史。可使用已经完成的梯度函数。报告测试准确率并解释至少一张错误数字；不同设置不保证更高分。

In [ ]:
def independent_training():
    # TODO: 使用自己实现的梯度函数和 train 完成变式，返回 (参数,损失历史)
    return None

In [ ]:
changed=independent_training()
if changed is None:print('待完成：独立训练变式。')
else:check_training(changed)

## 提示与复盘

先检查输入形状、数据类型和边界情况。网页“查看提示”提供针对本实验的提示。记录你遇到的一处错误、修复方法和结果。通过数值检查后，还需要自己解释原理。

## 参考解答

先独立尝试再展开下方代码。示例是可核对的一种实现，不是唯一写法。

In [ ]:
check_gradients(demo_gradients)
check_training((network,history))
check_training(train(demo_gradients,hidden=16,seed=7))